In [18]:
import json
import os

import pandas as pd

from util.web import download

In [19]:
csv_file_districts = "" or "data/election_results_districts.csv"
url_districts = "https://bundeswahlleiterin.de/dam/jcr/62868510-b616-443c-97b4-71847916d543/btw2021_brief_wkr.csv"
output_direktmandate = "data/election_results_direktmandate.json"
if not os.path.exists(csv_file_districts):
    print("Downloading file...")
    download(url_districts, csv_file_districts)
else:
    print("Using existing local file.")

Using existing local file.


In [20]:
df_district = pd.read_csv(
    csv_file_districts,
    skiprows=4,
    encoding='utf-8',
    header=[0, 1],
    sep=';'
)

In [21]:
# Process the multi-level columns to remove "Unnamed" labels
df_district.columns = [
    top if (isinstance(bot, str) and (bot.startswith('Unnamed') or bot.strip() == ''))
    else f"{top}_{bot}"
    for top, bot in df_district.columns
]

# Optional: Clean up any trailing underscores from empty bottom headers (if needed)
df_district.columns = [col.strip('_') for col in df_district.columns]
df_district

,Wahlkreis-Nr.,Wahlkreisname,Land,Wahlbezirksart,Wahlberechtigte,Wählende,Ungültige_Erststimmen,Gültige_Erststimmen,CDU_Erststimmen,SPD_Erststimmen,...,III. Weg_Zweitstimmen,Bündnis21_Zweitstimmen,LIEBE_Zweitstimmen,LKR_Zweitstimmen,PdF_Zweitstimmen,LfK_Zweitstimmen,SSW_Zweitstimmen,Team Todenhöfer_Zweitstimmen,UNABHÄNGIGE_Zweitstimmen,Volt_Zweitstimmen
0,1,Flensburg – Schleswig,SH,Urne,231536,127698,1270,126428,29743,29588,...,0,0,0,45,0,0,12167,238,0,262
1,2,Nordfriesland – Dithmarschen Nord,SH,Urne,188267,97703,1002,96701,29742,27360,...,0,0,0,37,0,0,6579,142,0,143
2,3,Steinburg – Dithmarschen Süd,SH,Urne,176899,98813,1037,97776,28706,28769,...,0,0,0,51,0,0,2436,211,0,164
3,4,Rendsburg-Eckernförde,SH,Urne,202226,111968,1092,110876,32549,34891,...,0,0,0,33,0,0,5185,182,0,167
4,5,Kiel,SH,Urne,202482,93891,853,93038,16306,28564,...,0,0,0,35,0,0,2655,585,0,323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,295,Zollernalb – Sigmaringen,BW,Brief,0,63816,595,63221,19529,11444,...,0,24,0,15,0,0,0,167,0,132
594,296,Saarbrücken,SL,Brief,0,63308,909,62399,17033,22776,...,0,0,0,0,0,0,0,287,0,755
595,297,Saarlouis,SL,Brief,0,59031,1068,57963,17322,21430,...,0,0,0,0,0,0,0,201,0,330
596,298,St. Wendel,SL,Brief,0,54160,1106,53054,17683,18988,...,0,0,0,0,0,0,0,188,0,264


In [22]:
df_aggregated = (df_district
                 .drop(columns=["Wahlberechtigte", "Wählende", "Ungültige_Erststimmen", "Gültige_Erststimmen"]
                               + [c for c in df_district.columns if c.endswith("Zweitstimmen")])
                 .groupby(["Wahlkreis-Nr.", "Wahlkreisname", "Land"])
                 .sum(numeric_only=True).reset_index())
df_aggregated

,Wahlkreis-Nr.,Wahlkreisname,Land,CDU_Erststimmen,SPD_Erststimmen,AfD_Erststimmen,FDP_Erststimmen,DIE LINKE_Erststimmen,GRÜNE_Erststimmen,CSU_Erststimmen,...,UNABHÄNGIGE_Erststimmen,Volt_Erststimmen,Volksabstimmung_Erststimmen,B*_Erststimmen,sonstige_Erststimmen,FAMILIE_Erststimmen,Graue Panther_Erststimmen,KlimalisteBW_Erststimmen,THP_Erststimmen,Übrige_Erststimmen
0,1,Flensburg – Schleswig,SH,41721,38927,9768,12299,6544,50231,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Nordfriesland – Dithmarschen Nord,SH,43745,40026,8274,13958,4060,20611,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Steinburg – Dithmarschen Süd,SH,39431,39379,10429,19315,4339,16686,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Rendsburg-Eckernförde,SH,47688,49474,10200,12903,4416,23832,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Kiel,SH,28416,45709,7147,11445,7275,43532,0,...,0,666,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,295,Zollernalb – Sigmaringen,BW,41106,24970,15554,18678,2848,23118,0,...,0,0,0,0,0,0,0,0,0,0
295,296,Saarbrücken,SL,35252,51749,12569,11647,8711,11143,0,...,0,0,0,0,0,0,0,0,0,299
296,297,Saarlouis,SL,43671,57354,15111,12777,8405,7383,0,...,0,0,0,0,0,0,0,0,0,0
297,298,St. Wendel,SL,43928,48135,12346,11354,5783,5739,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
erststimmen_columns = [
    col for col in df_aggregated.columns
    if col.endswith("_Erststimmen")  # Ensure we only include party votes
]

df_aggregated["Direktmandat_Winner"] = (
    df_aggregated[erststimmen_columns]
    .idxmax(axis=1)  # Get column name of the max value per row
    .str.split("_")  # Split column name (e.g., "CDU_Erststimmen")
    .str[0]
)

df_aggregated[["Wahlkreisname", "Direktmandat_Winner"]]

,Wahlkreisname,Direktmandat_Winner
0,Flensburg – Schleswig,GRÜNE
1,Nordfriesland – Dithmarschen Nord,CDU
2,Steinburg – Dithmarschen Süd,CDU
3,Rendsburg-Eckernförde,SPD
4,Kiel,SPD
...,...,...
294,Zollernalb – Sigmaringen,CDU
295,Saarbrücken,SPD
296,Saarlouis,SPD
297,St. Wendel,SPD


In [24]:
df_district

,Wahlkreis-Nr.,Wahlkreisname,Land,Wahlbezirksart,Wahlberechtigte,Wählende,Ungültige_Erststimmen,Gültige_Erststimmen,CDU_Erststimmen,SPD_Erststimmen,...,III. Weg_Zweitstimmen,Bündnis21_Zweitstimmen,LIEBE_Zweitstimmen,LKR_Zweitstimmen,PdF_Zweitstimmen,LfK_Zweitstimmen,SSW_Zweitstimmen,Team Todenhöfer_Zweitstimmen,UNABHÄNGIGE_Zweitstimmen,Volt_Zweitstimmen
0,1,Flensburg – Schleswig,SH,Urne,231536,127698,1270,126428,29743,29588,...,0,0,0,45,0,0,12167,238,0,262
1,2,Nordfriesland – Dithmarschen Nord,SH,Urne,188267,97703,1002,96701,29742,27360,...,0,0,0,37,0,0,6579,142,0,143
2,3,Steinburg – Dithmarschen Süd,SH,Urne,176899,98813,1037,97776,28706,28769,...,0,0,0,51,0,0,2436,211,0,164
3,4,Rendsburg-Eckernförde,SH,Urne,202226,111968,1092,110876,32549,34891,...,0,0,0,33,0,0,5185,182,0,167
4,5,Kiel,SH,Urne,202482,93891,853,93038,16306,28564,...,0,0,0,35,0,0,2655,585,0,323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,295,Zollernalb – Sigmaringen,BW,Brief,0,63816,595,63221,19529,11444,...,0,24,0,15,0,0,0,167,0,132
594,296,Saarbrücken,SL,Brief,0,63308,909,62399,17033,22776,...,0,0,0,0,0,0,0,287,0,755
595,297,Saarlouis,SL,Brief,0,59031,1068,57963,17322,21430,...,0,0,0,0,0,0,0,201,0,330
596,298,St. Wendel,SL,Brief,0,54160,1106,53054,17683,18988,...,0,0,0,0,0,0,0,188,0,264


In [25]:
district_winners = df_aggregated["Direktmandat_Winner"].value_counts().reset_index()
district_winners.columns = ["party", "districts_won"]
result_json = district_winners.to_dict(orient="records")

print(json.dumps(result_json, indent=2, ensure_ascii=False))

[
  {
    "party": "SPD",
    "districts_won": 121
  },
  {
    "party": "CDU",
    "districts_won": 98
  },
  {
    "party": "CSU",
    "districts_won": 45
  },
  {
    "party": "GRÜNE",
    "districts_won": 16
  },
  {
    "party": "AfD",
    "districts_won": 16
  },
  {
    "party": "DIE LINKE",
    "districts_won": 3
  }
]


In [26]:
with open(output_direktmandate, 'w') as f:
    json.dump(result_json, f, indent=4)